# Week 05: LLM basics, IPO prompts & joining tables

**Course:** MATH7810 Python for Data Science (AY2026/27)  
**Environment:** GitHub Codespaces + GitHub Copilot (**Inline Chat**)

**Data (trimmed teaching extract):** from the [NBA AB-Index](https://github.com/taliawu17/NBA_AB-index) project  
- `data/nba_players_bio_trim.csv` — player bio (50 players)  
- `data/nba_career_summary_trim.csv` — career summary for the **same** players  

**Class path:** LLM + IPO → **research questions** → load & explore → Inline Chat demo → join → Q1/Q2 → short write-up.

**Your responsibility:** AI only drafts. **You** own the final work — read the code, run it, and check that the output answers the question. Do **not** accept every AI output without verification.


## A. LLM basics & IPO prompts

### What is an LLM?
An **LLM** (Large Language Model) predicts likely next **tokens**. Fluent code or English can still be **wrong**. Treat AI as a **draft partner**, not an authority.

| Idea | For you |
|------|---------|
| Probability-based | Answers are not guaranteed facts |
| Prompt → quality | Vague prompts may “work” but miss *your* constraints |
| Human-in-the-loop | You propose → AI drafts → you verify |

### IPO prompt pattern 
Write prompts as **Input → Process → Output**. 

| Part | Meaning | Example |
|------|---------|---------|
| **Input** | What already exists (dataframes, columns) | `bio` and `career` already loaded |
| **Process** | One clear job | Write Python code to merge on `personId` |
| **Output** | What you must see | A joined table named `players` |

### How to use Inline Chat in Codespaces
1. Click inside the code cell.  
2. Keep / edit the comment prompt.  
3. Press **`Cmd+I`** (Mac) or **`Ctrl+I`** (Windows/Linux).  
4. Ask e.g. *Implement the prompt in the comments above* → **Keep**.  
5. **Read** the code, fix if needed, then run. Write interpretations yourself.


## B. Research questions first

In real analysis — and in your **case study** — you do **not** start by asking AI for “some code.” You start with a **research question**. The question decides which tables you need and what to filter or summarize.

**Today’s questions (this 50-player sample only):**

1. **In this sample, how does mean retirement age (`retire_age`) compare across `position`?**  
2. **Among players with longer careers (`career_years >= 5`), which positions appear most often in this sample?**

**Why two files?**  
- **Bio** has `position` (and other player attributes).  
- **Career** has `retire_age` and `career_years`.  
Neither file alone can answer Q1 or Q2. You need a shared key (**`personId`**) and a **join**, then a summary (Q1) or a filter + counts (Q2).

Use **association** language only. This is a **teaching sample**, not the whole NBA — do not claim causation or “all players worldwide.”


## Setup — run once
Load both CSVs. Glance at column names.

In [1]:
# Setup — run this cell first
import pandas as pd
import numpy as np
from pathlib import Path

bio_path = Path("data/nba_players_bio_trim.csv")
career_path = Path("data/nba_career_summary_trim.csv")

bio = pd.read_csv(bio_path)
career = pd.read_csv(career_path)

print("bio:", bio.shape, list(bio.columns))
print("career:", career.shape, list(career.columns))
print(bio.head(3))
print(career.head(3))


bio: (50, 10) ['personId', 'PlayerName', 'position', 'country', 'draftYear', 'height_in', 'weight_lb', 'guard', 'forward', 'center']
career: (50, 6) ['personId', 'player_name', 'season_from', 'season_to', 'retire_age', 'career_years']
   personId       PlayerName position country  draftYear  height_in  \
0        29    Jim McIlvaine        C     USA     1994.0        NaN   
1        30    Richard Dumas        F     USA     1991.0       79.0   
2        32  Clifford Rozier      C-F     USA     1994.0       83.0   

   weight_lb  guard forward center  
0        NaN  False   False   True  
1       30.0  False    True  False  
2      245.0  False   False   True  
   personId      player_name  season_from  season_to  retire_age  career_years
0        29    Jim McIlvaine         1995       2001          29             6
1        30    Richard Dumas         1993       1996          27             3
2        32  Clifford Rozier         1995       1998          26             3


## Task 1 — Explore each table

Before joining, check that both files look usable: shapes, missing values, and whether `personId` looks like a shared key.


In [2]:
# Task 1 — Explore
# --- Sample AI prompt (Input → Process → Output) ---
# Input: dataframes bio and career already loaded.
# Process: Write Python code to print shape, column names, missing-value counts for bio and for career. 
#   Also print how many unique personId values are in
#   each table and how many personId values appear in BOTH tables.
#   Add short comments to key lines.
# Output: Printed summaries (shapes, na counts, unique / shared personId counts).
# --- End prompt ---
# Use Inline Chat (Cmd+I / Ctrl+I), then read the code before running.
#
# --- generated / your code below ---
for name, df in {"bio": bio, "career": career}.items():
    print(f"\n{name} shape: {df.shape}")
    print(f"{name} columns: {list(df.columns)}")
    print(f"{name} missing values:\n{df.isna().sum()}")

bio_ids = set(bio["personId"].dropna())
career_ids = set(career["personId"].dropna())
print(f"\nUnique bio personId values: {bio['personId'].nunique()}")
print(f"Unique career personId values: {career['personId'].nunique()}")
print(f"personId values in both tables: {len(bio_ids & career_ids)}")



bio shape: (50, 10)
bio columns: ['personId', 'PlayerName', 'position', 'country', 'draftYear', 'height_in', 'weight_lb', 'guard', 'forward', 'center']
bio missing values:
personId      0
PlayerName    0
position      0
country       1
draftYear     1
height_in     4
weight_lb     4
guard         2
forward       2
center        2
dtype: int64

career shape: (50, 6)
career columns: ['personId', 'player_name', 'season_from', 'season_to', 'retire_age', 'career_years']
career missing values:
personId        0
player_name     0
season_from     0
season_to       0
retire_age      0
career_years    0
dtype: int64

Unique bio personId values: 50
Unique career personId values: 50
personId values in both tables: 50


## Task 2 — Bio only: height & weight by position

Before we join, practice IPO + Inline Chat on **`bio` alone**.

**Goal:** for each **`position`**, compute the **mean** and **standard deviation** of **`height_in`** and **`weight_lb`**.

The comment prompt below is ready. Use **Inline Chat** (`Cmd+I` / `Ctrl+I`) to generate the code — do **not** type the analysis code yourself first. Then **read** the draft, fix if needed, and run.


In [3]:
# Task 2 — Bio only (Inline Chat demo)
# --- Sample AI prompt (Input → Process → Output) ---
# Input: dataframe bio already loaded.
# Process: Write Python code to group bio by position and calculate the mean and standard deviation of height_in and weight_lb for each position.
# Output: A summary table by position with mean/SD for height_in and weight_lb.
# --- End prompt ---
# Use Inline Chat (Cmd+I / Ctrl+I): ask to implement the prompt in the comments.
# Read the generated code before running.
#
# --- generated / your code below ---
bio_by_position = bio.groupby("position").agg(
    height_mean=("height_in", "mean"),
    height_sd=("height_in", "std"),
    weight_mean=("weight_lb", "mean"),
    weight_sd=("weight_lb", "std"),
).sort_index()
print(bio_by_position)


          height_mean  height_sd  weight_mean  weight_sd
position                                                
C           85.000000   2.183857   265.142857  26.645413
C-F         83.000000   0.000000   247.500000   3.535534
F           80.363636   1.747726   210.454545  62.576934
F-C         83.000000   1.000000   265.000000   8.660254
F-G         80.000000        NaN   205.000000        NaN
G           75.500000   1.990361   192.071429  13.111735
G-F         78.000000        NaN   210.000000        NaN


## Task 3 — Join the tables

Merge **bio** and **career** on **`personId`** so each row is one player with bio + career fields together.

**Your job:** complete **Process** and **Output** in the comment prompt (use the task description above). Keep **Process** starting with **Write Python code**. Then use Inline Chat.


In [4]:
# Task 3 — Merge
# --- Write your AI prompt here (Input → Process → Output) ---
# Input: dataframes bio and career already loaded; both have personId.
# Process: Write Python code to merge bio and career on personId with an inner join, validate that the key is one-to-one, and store the result as players.
# Output: A joined dataframe named players with one row per player and bio plus career columns.
# --- End prompt ---
# Fill Process + Output from the task description, then use Inline Chat.
#
# --- generated / your code below ---
players = bio.merge(
    career,
    on="personId",
    how="inner",
    validate="one_to_one",
)
print(players.shape)
print(players.head())


(50, 15)
   personId       PlayerName position country  draftYear  height_in  \
0        29    Jim McIlvaine        C     USA     1994.0        NaN   
1        30    Richard Dumas        F     USA     1991.0       79.0   
2        32  Clifford Rozier      C-F     USA     1994.0       83.0   
3        35       Eric Riley        C     USA     1993.0       84.0   
4        37      Greg Graham        G     USA     1993.0       76.0   

   weight_lb  guard forward center      player_name  season_from  season_to  \
0        NaN  False   False   True    Jim McIlvaine         1995       2001   
1       30.0  False    True  False    Richard Dumas         1993       1996   
2      245.0  False   False   True  Clifford Rozier         1995       1998   
3      245.0  False   False   True       Eric Riley         1994       1999   
4      174.0   True   False  False      Greg Graham         1994       1998   

   retire_age  career_years  
0          29             6  
1          27             3  

## Task 4 — Answer **Q1** (group summary)

**Q1:** In this sample, how does mean retirement age (`retire_age`) compare across `position`?

For each **position**, report how many players and the **mean retire_age**. Sort so the largest groups appear first.

**Your job:** complete **Process** and **Output** in the comment prompt. Keep **Process** starting with **Write Python code**. Then use Inline Chat.


In [5]:
# Task 4 — Group summary
# --- Write your AI prompt here (Input → Process → Output) ---
# Input: dataframe players already created (joined bio + career).
# Process: Write Python code to group players by position, report the number of players and mean retire_age for each position, and sort by group size from largest to smallest.
# Output: A table with position, player count, and mean retire_age, sorted by player count descending.
# --- End prompt ---
# Fill Process + Output from the task description, then use Inline Chat.
#
# --- generated / your code below ---
q1_summary = (
    players.groupby("position")
    .agg(
        n_players=("personId", "size"),
        mean_retire_age=("retire_age", "mean"),
    )
    .sort_values("n_players", ascending=False)
)
print(q1_summary)


          n_players  mean_retire_age
position                            
C                16        27.625000
G                14        27.571429
F                13        27.153846
F-C               3        26.666667
C-F               2        26.500000
F-G               1        28.000000
G-F               1        28.000000


## Task 5 — Answer **Q2** (filter + counts)

**Q2:** Among players with longer careers (`career_years >= 5`), which positions appear most often in this sample?

Keep only players with **`career_years >= 5`**. Among them, show **position** counts (or a simple bar of counts by position). Then answer in your own words below.


In [6]:
# Task 5 — Filter long careers
# --- Sample AI prompt (Input → Process → Output) ---
# Input: dataframe players already created.
# Process: Write Python code to keep players with career_years >= 5 and count the remaining rows by position.
# Output: A position count table for players with career_years >= 5, ordered from most common to least common.
# --- End prompt ---
#
# --- generated / your code below ---
long_careers = players.loc[players["career_years"] >= 5]
q2_counts = long_careers["position"].value_counts()
print(q2_counts)


position
C      9
F      6
G      5
F-G    1
F-C    1
Name: count, dtype: int64


### Task 5 — Your words on **Q1 / Q2** (2–4 sentences)

In this **sample** (not the whole NBA): what do Q1 and Q2 suggest about **position**, retirement age, and longer careers? Use **association** language only. Mention if some positions have very small *n*.

**How to answer:** type inside the triple quotes in the next cell, then **run** it.


In [7]:
# Task 5 — write BETWEEN the triple quotes, then RUN this cell
task5_note = """
# What do Q1 and Q2 suggest in this sample?

Q1 shows that the largest position groups are C (n=16), G (n=14), and F (n=13), with mean retirement ages of about 27.6, 27.6, and 27.2 years, respectively. Q2 shows that among players with at least five career years, C appears most often (9), followed by F (6) and G (5). The small F-G (n=1), F-C (n=3), and C-F (n=2) groups make their mean retirement ages less stable, so these results describe associations in this sample rather than causal effects or the whole NBA.
"""
print(task5_note)



# What do Q1 and Q2 suggest in this sample?

Q1 shows that the largest position groups are C (n=16), G (n=14), and F (n=13), with mean retirement ages of about 27.6, 27.6, and 27.2 years, respectively. Q2 shows that among players with at least five career years, C appears most often (9), followed by F (6) and G (5). The small F-G (n=1), F-C (n=3), and C-F (n=2) groups make their mean retirement ages less stable, so these results describe associations in this sample rather than causal effects or the whole NBA.



**After class:** try one **own** IPO prompt to compare mean `career_years` for `guard == True` vs others on `players`.


### Data credit
Trimmed for MATH7810 teaching from [taliawu17/NBA_AB-index](https://github.com/taliawu17/NBA_AB-index) (`Players.csv` + career target list with `personId`). Full pipeline / AB-index ranking is **not** required in this lab.
